# 13 · Second dataset: LFP first- and second-life ageing (S6, protocol D20)

**Data.** Wheeler et al. 2025 (*Scientific Data* 12, 392; data doi:10.57745/OLBXKT, CC BY 4.0; `docs/EXTERNAL_DATA.md` S6):
- 20 A123 18650 Graphite/LFP cells in seven test cases (usage conditions);
- aged at 50 °C, with a characterisation every 100 cycles, down to 30–50 % SOH.

In S6, "second life" means ageing below 80 % SOH under the same protocol, not a repurposed duty.

**Pipeline.**
- `scripts/fetch_wheeler_characterisation.py` fetches the characterisation files (checksummed).
- `scripts/build_wheeler_features.py` builds one row per (cell, check) with C/25 SOH labels (D17 analogue) and diagnostic features, and checks the parsing against the authors' `extractedData.mat`.
- This notebook applies the forecasting protocol D20: D18 with test cases as tasks.

## 1. Checks and SOH trajectories

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

import torch
torch.set_num_threads(1)
from src import experiments as X
from src import evaluation as ev
from src import forecasting as FC
from src import wheeler as W

dp.check_data_files(DATA_ROOT, required=["wheeler_checks.csv"])
checks = dp.read_dataset(DATA_ROOT / "wheeler_checks.csv")
checks["check_time"] = pd.to_datetime(checks["check_time"])
CASES = sorted(checks[dp.GROUP_COL].unique())
CASE_COLORS = dict(zip(CASES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]))
CASE_MARKERS = dict(zip(CASES, ["o", "s", "^", "D", "v", "P", "X"]))
summary = checks.groupby([dp.GROUP_COL, "cell"]).agg(checks=("cycle", "size"), valid=("valid", "sum"),
                                                    last_cycle=("cycle", "max"), soh_end=("SOH_Percent", "last"))
display(summary.round(1))
print("invalid checks (no complete C/25 discharge):", checks.loc[~checks["valid"], ["cell", "cycle"]].values.tolist())
display(W.TEST_CASES)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
for case in CASES:
    for i, (cell, g) in enumerate(checks[checks["valid"] & (checks[dp.GROUP_COL] == case)].groupby("cell")):
        ax.plot(g["cycle"], g["SOH_Percent"], marker=CASE_MARKERS[case], markersize=3, linewidth=1.2,
                color=CASE_COLORS[case], label=case if i == 0 else None)
for y, lab in ((95, "C0/C1"), (90, "C1/C2"), (80, "C2/C3")):
    ax.axhline(y, color="#8a8981", linewidth=0.8, linestyle="--")
    ax.text(ax.get_xlim()[1], y, f" {lab}", va="center", fontsize=8, color="#3d3d3a")
ax.set_xlabel("Ageing cycles"); ax.set_ylabel("SOH, C/25 discharge capacity (% of first check)")
ax.set_title("S6: SOH of 20 LFP cells by test case (reuse-class cut-points dashed)", loc="left")
ax.legend(frameon=False, fontsize=8, ncol=4)
plt.tight_layout()
savefig(fig, "13_soh_trajectories")
plt.show()

In [ ]:
# IC curves of one cell per case at every 5th check (light = early).
raw = DATA_ROOT / W.RAW_SUBDIR
lst = W.list_checks(raw)
fig, axes = plt.subplots(2, 4, figsize=(13, 6), sharex=True, sharey=True)
cmap = plt.get_cmap("Blues")
for ax, case in zip(axes.flat, CASES):
    cell = sorted(lst.loc[lst[dp.GROUP_COL] == case, "cell"].unique())[0]
    sub = lst[lst["cell"] == cell].iloc[::5]
    for k, (_, r) in enumerate(sub.iterrows()):
        ts = W.read_check(r["path"])
        d25 = W._step(ts, 16, True)
        if len(d25):
            ax.plot(W.IC_GRID, W.ic_curve(d25), color=cmap(0.3 + 0.7 * k / max(len(sub) - 1, 1)), linewidth=1)
    ax.set_title(f"{case} ({cell})", loc="left", fontsize=9)
    ax.set_xlim(3.15, 3.38)
axes.flat[-1].axis("off")
for ax in axes[-1]:
    ax.set_xlabel("Voltage (V)")
for ax in axes[:, 0]:
    ax.set_ylabel("−dQ/dV (Ah/V)")
fig.suptitle("S6: incremental capacity of the C/25 discharge (every 5th check)", x=0.02, ha="left")
plt.tight_layout()
savefig(fig, "13_ic_curves")
plt.show()

## 2. Forecasting samples and splits

In [ ]:
samples = W.build_samples(checks)
SETS = W.feature_sets(samples)
split = FC.assign_split(samples)
print(f"{len(samples)} samples; " + ", ".join(f"{k}={len(v)}" for k, v in SETS.items()))
display(pd.crosstab(split[dp.GROUP_COL], split["split"]))

## 3. Tuning (validation only; D11 with the extended learning-rate grid, D20)

In [ ]:
# EXPERIMENT_PROFILE=smoke runs a tiny configuration to check the code end to end; only "paper" results are reportable.
PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
LR_GRID = [3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
if PROFILE == "smoke":
    base_cfg = FC.cl_config(SETS["F1"], n_orders=1, seeds=(0,))
    base_cfg.common_kwargs["epochs"] = 2
    tcfg = X.TuningConfig(backbone_grid={"lr": [1e-3]}, grids={"ewc": {"ewc_lambda": [10.0]}}, n_orders=1)
else:
    base_cfg = FC.cl_config(SETS["F1"])
    tcfg = X.TuningConfig(backbone_grid={"lr": LR_GRID})
print("profile:", PROFILE)
tuning = X.tune_hyperparameters(samples, base_cfg, tcfg, stream_builder=FC.forecast_stream)
CFG = tuning["tuned_cl_config"]
print("backbone:", tuning["backbone_selected"]); print("methods:", tuning["selected"]); print("at grid edge:", tuning["at_grid_edge"])

## 4. P1: continual stream over the seven test cases

In [ ]:
p1 = FC.run_p1(samples, CFG)
cfg_f0 = FC.cl_config(SETS["F0"], methods=("tft_ewc",), n_orders=CFG.n_orders, seeds=CFG.seeds)
cfg_f0.common_kwargs.update({k: v for k, v in CFG.common_kwargs.items() if k not in ("features",)})
cfg_f0.backbone_kwargs, cfg_f0.method_kwargs = CFG.backbone_kwargs, CFG.method_kwargs
p1_f0 = FC.run_p1(samples, cfg_f0, label="_F0")
runs = pd.concat([p1["runs"], p1_f0["runs"]], ignore_index=True)
final = pd.concat([p1["final_per_task"], p1_f0["final_per_task"]], ignore_index=True)
joint = FC.run_joint(samples, CFG)
base_p1 = FC.run_baselines_p1(samples, SETS, columns=W.BASELINE_COLUMNS)
cl_summary = ev.summarise_runs(runs, ["final_avg_error", "plasticity", "backward_transfer", "forgetting", "forward_transfer"], ["method"])
display(cl_summary.round(3))

def per_method(frame, col="RMSE"):
    keys = [c for c in ("order_id", "seed") if c in frame]
    return frame.groupby(["method", *keys])[col].mean().groupby("method").agg(["mean", "std", "size"])

rmse_table = pd.concat([per_method(final), per_method(joint), per_method(base_p1.assign(seed=0))]).sort_values("mean")
persist = rmse_table.loc["persistence", "mean"]
rmse_table["skill_vs_persistence"] = 1 - rmse_table["mean"] / persist
cls = pd.concat([final, joint, base_p1]).groupby("method")[["class_accuracy", "class_macro_f1"]].mean()
rmse_table = rmse_table.join(cls)
display(rmse_table.round(3))
h_cols = [c for c in final.columns if c.startswith("RMSE_h")]
by_h = pd.concat([final.groupby("method")[h_cols].mean(), joint.groupby("method")[h_cols].mean(), base_p1.groupby("method")[h_cols].mean()])
display(by_h.loc[rmse_table.index].round(3))

In [ ]:
comp = X.compare_methods(runs, baseline="tft_ewc", metric_cols=["final_avg_error", "forgetting", "backward_transfer"])
display(comp[["metric", "method", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm", "rank_biserial"]].round(4))
tft = runs[runs["method"] == "tft_ewc"].sort_values(["order_id", "seed"])["final_avg_error"].to_numpy()
base_final = base_p1.groupby("method")["RMSE"].mean()
vs_base, pv = [], {}
for name, value in base_final.items():
    res = ev.paired_comparison(tft, np.full(len(tft), value))
    vs_base.append({"baseline": name, "baseline_final_avg_error": value, **res}); pv[name] = res["p_value"]
holm = ev.holm_correction(pv)
vs_base = pd.DataFrame(vs_base).assign(p_holm=lambda d: d["baseline"].map(holm))
display(vs_base.round(4))

In [ ]:
order = list(rmse_table.index)
fig, ax = plt.subplots(figsize=(7.5, 4.4))
colors = ["#2a78d6" if m.startswith("tft") else "#8a8981" if m in base_final.index else "#1baf7a" for m in order]
ax.barh(order, rmse_table["mean"], xerr=rmse_table["std"].fillna(0), color=colors, height=0.6, error_kw={"elinewidth": 1})
for i, v in enumerate(rmse_table["mean"]):
    ax.text(v, i, f"  {v:.2f}", va="center", fontsize=8, color="#3d3d3a")
ax.axvline(persist, color="#1a1a19", linewidth=1, linestyle="--")
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("Test RMSE of SOH(k+h), pp (mean over test cases; ±1 sd over runs)")
ax.set_title("S6, P1: final-step forecast error (blue: TFT, green: MLP, grey: baselines)", loc="left")
plt.tight_layout(); savefig(fig, "13_p1_rmse"); plt.show()

mats = [m.to_numpy(dtype=float) for (name, _, _), m in p1["matrices"].items() if name == "tft_ewc"]
M = np.nanmean(np.stack(mats), axis=0)
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(M, cmap="Blues", vmin=0)
for (i, j), v in np.ndenumerate(M):
    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="white" if v > np.nanmax(M) * 0.6 else "#1a1a19")
ax.set_xticks(range(M.shape[1]), range(1, M.shape[1] + 1)); ax.set_yticks(range(M.shape[0]), range(1, M.shape[0] + 1))
ax.set_xlabel("Task position j"); ax.set_ylabel("After training task i"); ax.grid(False)
ax.set_title("S6, TFT + EWC: test RMSE (pp), mean of 50 runs", loc="left", fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.8); savefig(fig, "13_tft_ewc_matrix"); plt.show()

## 5. P2: unseen usage condition (leave one test case out)

In [ ]:
p2 = FC.run_p2(samples, CFG, SETS, columns=W.BASELINE_COLUMNS)
p2_case = p2.groupby(["method", dp.GROUP_COL])["RMSE"].mean().unstack()
p2_table = p2_case.assign(mean=p2_case.mean(axis=1)).sort_values("mean")
p2_table = p2_table.join(p2.groupby("method")[["class_accuracy", "class_macro_f1"]].mean())
display(p2_table.round(3))
pv2, rows2 = {}, []
for m in p2_case.index.drop("tft"):
    res = ev.paired_comparison(p2_case.loc["tft"].to_numpy(), p2_case.loc[m].to_numpy())
    rows2.append({"method": m, **res}); pv2[m] = res["p_value"]
h2 = ev.holm_correction(pv2)
p2_tests = pd.DataFrame(rows2).assign(p_holm=lambda d: d["method"].map(h2))
display(p2_tests.round(4))

## 6. Save results

In [ ]:
out = X.write_results(RESULTS_DIR / ("forecasting_wheeler.json" if PROFILE == "paper" else "forecasting_wheeler_smoke.json"), {
    "profile": PROFILE,
    "dataset": "S6, Wheeler et al. 2025 (doi:10.57745/OLBXKT)", "protocol": "docs/DECISIONS.md D20",
    "n_checks": int(len(checks)), "n_valid_checks": int(checks["valid"].sum()), "n_samples": int(len(samples)),
    "feature_sets": {k: list(v) for k, v in SETS.items()},
    "tuning": {"backbone_selected": tuning["backbone_selected"], "selected": tuning["selected"], "at_grid_edge": tuning["at_grid_edge"]},
    "p1_cl_summary": cl_summary, "p1_rmse": rmse_table.reset_index(names="method"),
    "p1_rmse_by_horizon": by_h.reset_index(names="method"), "p1_tests_vs_tft_ewc": comp,
    "p1_tft_ewc_vs_baselines": vs_base, "p2_rmse": p2_table.reset_index(names="method"), "p2_tests_vs_tft": p2_tests,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT))

## 7. Interpretation

Written from the executed run in `docs/DATA_REPORT.md` §8; numbers in `results/forecasting_wheeler.json`.